In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

from sliced_wasserstein import sliced_wasserstein_distance
from c2st import c2st_knn, c2st_nn, c2st_rf

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
import pickle

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 2
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [ ]:
SUBJECT_INDEX = 2

In [ ]:

def load_model(cfg, start_index=100, subject_index=2):
    save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/model_checkpoints/finetune"
    
    file_path = os.path.join(save_path, f"subject_{subject_index}", f"model_checkpoint_finetune_subject_index_{subject_index}_start_idx_{start_index}_rep_0_pen.pth") 
    trunk_net = TrunkNet(n_chans=input_shape_st[0], n_times=input_shape_st[1])
    head_net = HeadNet(64, 1)  # Assuming these are the correct dimensions
    model = S4PatchedFinalNet(64, trunk_net, head_net)
    
    weights = torch.load(file_path)
    model.load_state_dict(weights)
    #model.load_state_dict(full_checkpoint['model_state_dict'])
    model.eval()
    model.to(device)
    return model

In [ ]:
freq_bands = {
              "delta": (0, 4),
            "theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}
phase_peturbations = np.arange(45, 316, 45)

In [ ]:
import matplotlib.pylab as pylab
params = {'legend.fontsize': 'x-large',
          'figure.titlesize': 'x-large',
          'figure.figsize': (15, 5),
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'x-large',
         'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

In [ ]:
def load_predicted_amplitude_for_subject(subject_index=2, rep=1):
    data_dir = f"/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/gradshap_explanations_rep_{rep}"
    file_path = os.path.join(data_dir, f"gradshap_data_subject_{subject_index}_rep_{rep}.npy")

    subject_data = np.load(file_path, allow_pickle=True).item()
    predictions, uncertainties, explanations = subject_data['predictions'], subject_data['uncertainties'], subject_data['explanations']

        
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
    epochs = mne.read_epochs(file_path)
    info = epochs.info
    
    return predictions, uncertainties, explanations, ch_names, info

# boilerplate

In [ ]:
pred_label_original,_,_,ch_names, info = load_predicted_amplitude_for_subject(2, rep=1)

In [ ]:



def load_distances(subject_index, factors, rep=1):
    dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_phase/parallel_perturbation_distance_phase"
    
    distances = {}

    for band_name, (low_freq, high_freq) in freq_bands.items():
        distances[band_name] = {}

        for factor in factors:
            file_path = f"parallel_distance_dict_{band_name}_channel_phase_shift_{factor}°_subject_{subject_index}_rep_{rep}.npy"
            load_path = os.path.join(dir, file_path)
            distances[band_name][factor] = np.load(load_path, allow_pickle=True).item()

    return distances

In [ ]:

def calculate_diff_per_channel_dw(pred_label_original, freq_bands, amplification_factors, ch_names, distances, subject_index=2, rep=1, take_abs):
    dir_constrained = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_phase/parallel_perturbation_phase"
    mean_diff_per_channel = {}
    median_diff_per_channel = {}

    for band_name, (low_freq, high_freq) in freq_bands.items():
        mean_diff_per_channel[band_name] = {}
        median_diff_per_channel[band_name] = {}
        for factor in amplification_factors:
            file_path = f"parallel_perturbed_prediction_dict_{band_name}_channel_phase_shift_{factor}°_subject_{subject_index}_rep_{rep}.npy"
            load_path = os.path.join(dir_constrained, file_path)
            perturbed_data = np.load(load_path, allow_pickle=True).item()
            mean_diff_per_channel[band_name][factor] = {}
            median_diff_per_channel[band_name][factor] = {}
            for ch_name in ch_names:
                perturbed_amplitude = np.array(perturbed_data[ch_name])
                diff = np.abs(pred_label_original - perturbed_amplitude)/np.array(distances[band_name][factor][ch_name])
                mean_diff_per_channel[band_name][factor][ch_name] = np.mean(diff)
                median_diff_per_channel[band_name][factor][ch_name] = np.median(diff)
    
    return mean_diff_per_channel, median_diff_per_channel
  
#distance = load_distances(2, amplification_factors, rep=1)
#mean_diff_per_channel, median_diff_per_channel = calculate_diff_per_channel_dw(pred_label_original, freq_bands, #amplification_factors, ch_names, distance, subject_index=2, rep=1)


In [ ]:
def plot_channel_differences(median_diff_per_channel, freq_band, amp_factor, save_path=None):
    """
    Create a barplot of median differences for each channel for a specific frequency band and amplification factor.
    
    Parameters:
    -----------
    median_diff_per_channel : dict
        The dictionary containing median differences for each frequency band, amplification factor, and channel
    freq_band : str
        The frequency band to plot (e.g., 'theta', 'alpha', etc.)
    amp_factor : float
        The amplification factor to plot (e.g., 0.5, 1.5, etc.)
    save_path : str, optional
        Path to save the figure. If None, the figure is not saved.
    """
    # Extract data for the specified frequency band and amplification factor
    channel_diffs = median_diff_per_channel[freq_band][amp_factor]
    
    # Create DataFrame from the dictionary
    df = pd.DataFrame({
        'Channel': list(channel_diffs.keys()),
        'Median Difference': list(channel_diffs.values())
    })
    
    # Sort by median difference for better visualization
    df = df.sort_values('Median Difference', ascending=False)
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(12, 8))
    bars = ax.bar(df['Channel'], df['Median Difference'])
    
    # Add labels and title
    ax.set_xlabel('Channels')
    ax.set_ylabel('Median Absolute Difference')
    ax.set_title(f'Median Difference between Original and Perturbed Predictions\n{freq_band.capitalize()} Band, Amplification Factor: {amp_factor}')
    
    # Rotate x-axis labels for better readability
    plt.xticks(rotation=90)
    
    # Add grid for better readability
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    
    # Save the figure if a path is provided
    if save_path:
        plt.savefig(save_path)
        
    return fig, ax

# Example usage:
#plot_channel_differences(median_diff_per_channel, 'gamma', 2.0, 
#                         save_path=f"median_diff_alpha_factor_2_channels.png")



In [ ]:
from matplotlib.patches import Patch

def plot_channel_differences(median_diff_per_channel, freq_band, amp_factor, save_path=None):
    """
    Create a barplot of median differences for each channel for a specific frequency band and amplification factor.
    Channels are arranged according to their physical locations, not sorted by magnitude.
    
    Parameters:
    -----------
    median_diff_per_channel : dict
        The dictionary containing median differences for each frequency band, amplification factor, and channel
    freq_band : str
        The frequency band to plot (e.g., 'theta', 'alpha', etc.)
    amp_factor : float
        The amplification factor to plot (e.g., 0.5, 1.5, etc.)
    save_path : str, optional
        Path to save the figure. If None, the figure is not saved.
    """
    # Extract data for the specified frequency band and amplification factor
    channel_diffs = median_diff_per_channel[freq_band][amp_factor]
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(14, 5))
    
    # Create a set of important channels to highlight
    important_channels = {'C2','C4', 'C6', 'CP4', 'CP6', 'FC2', 'FC4', 'FC6'}
    
    # Filter channels to keep only important ones
    channels = [ch for ch in channel_diffs.keys() if ch in important_channels]
    values = [channel_diffs[ch] for ch in channels]
    
    # Sort the data by median difference for better visualization
    # Keep the original order of important channels
    sorted_values = []
    sorted_channels = []
    
    for ch in important_channels:
        if ch in channel_diffs:
            sorted_channels.append(ch)
            sorted_values.append(channel_diffs[ch])
    
    channels = sorted_channels
    values = sorted_values
    # Create the bar plot
    bars = ax.bar(channels, values)
    
    # Define regions and their corresponding colors
    frontal_channels = ['Fp1', 'Fp2']
    central_channels = ['C3', 'C4', 'Cz', 'C1', 'C2', 'C5', 'C6']
    parietal_channels = ['CP1', 'CP2', 'CP3', 'CP4', 'CP5', 'CP6', 'CPz']
    frontocentral_channels = ['FC1', 'FC2', 'FC3', 'FC4', 'FC5', 'FC6']
    
    # Color bars based on channel region
    for i, channel in enumerate(channels):
        if channel in frontal_channels:
            bars[i].set_color('red')
        elif channel in central_channels:
            bars[i].set_color('green')
        elif channel in parietal_channels:
            bars[i].set_color('blue')
        elif channel in frontocentral_channels:
            bars[i].set_color('brown')
    
    # Add labels and title
    ax.set_xlabel('Channels')
    ax.set_ylabel('Median Difference')
    ax.set_title(f'Median Difference between Original and Perturbed Predictions\n{freq_band.capitalize()} Band, Amplification Factor: {amp_factor}')
    
    # Rotate x-axis labels for better readability
    plt.xticks(rotation=90)
    
    # Add grid for better readability
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Add a legend for the regions
    legend_elements = [
        Patch(facecolor='red', label='Frontal'),
        Patch(facecolor='green', label='Central'),
        Patch(facecolor='purple', label='Temporal'),
        Patch(facecolor='blue', label='Parietal')]
        #Patch(face    # Keep the original order of important channels
    sorted_values = []
    sorted_channels = []
    
    for ch in important_channels:
        if ch in channel_diffs:
            sorted_channels.append(ch)
            sorted_values.append(channel_diffs[ch])
    
    channels = sorted_channels
    values = sorted_values()
    
    # Save the figure if a path is provided
    if save_path:
        plt.savefig(save_path)
        
    return fig, ax


## differenceces plots

In [ ]:
plot_channel_differences(median_diff_per_channel, 'theta', 2.0, 
                         save_path=f"median_diff_alpha_factor_2_channels.png")


In [ ]:
plot_channel_differences(median_diff_per_channel, 'alpha', 2.0, 
                         save_path=f"median_diff_alpha_factor_2_channels.png")

In [ ]:
plot_channel_differences(median_diff_per_channel, 'beta', 2.0, 
                         save_path=f"median_diff_alpha_factor_2_channels.png")

In [ ]:
plot_channel_differences(median_diff_per_channel, 'gamma', 2.0, 
                         save_path=f"median_diff_alpha_factor_2_channels.png")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def plot_channel_differences(median_diff_per_channel, freq_band, amp_factor=None, save_path=None):
    """
    Create a barplot of median differences for each channel for a specific frequency band.
    Multiple amplification factors are shown in separate subplots.
    
    Parameters:
    -----------
    median_diff_per_channel : dict
        The dictionary containing median differences for each frequency band, amplification factor, and channel
    freq_band : str
        The frequency band to plot (e.g., 'theta', 'alpha', etc.)
    amp_factor : float, optional
        If specified, plot only this amplification factor. If None, plot all.
    save_path : str, optional
        Path to save the figure. If None, the figure is not saved.
    """
    # Create a set of important channels to highlight
    important_channels = {'C2','C4', 'C6', 'CP4', 'CP6', 'FC2', 'FC4', 'FC6'}
    
    # Define regions and their corresponding colors
    frontal_channels = ['Fp1', 'Fp2']
    central_channels = ['C3', 'C4', 'Cz', 'C1', 'C2', 'C5', 'C6']
    parietal_channels = ['CP1', 'CP2', 'CP3', 'CP4', 'CP5', 'CP6', 'CPz']
    frontocentral_channels = ['FC1', 'FC2', 'FC3', 'FC4', 'FC5', 'FC6']
    
    if amp_factor is not None:
        # Original behavior for single amplification factor
        channel_diffs = median_diff_per_channel[freq_band][amp_factor]
        
        fig, ax = plt.subplots(figsize=(14, 5))
        
        # Filter and sort channels
        sorted_channels = [ch for ch in important_channels if ch in channel_diffs]
        sorted_values = [channel_diffs[ch] for ch in sorted_channels]
        
        # Create the bar plot
        bars = ax.bar(sorted_channels, sorted_values)
        
        # Color bars based on channel region
        for i, channel in enumerate(sorted_channels):
            if channel in frontal_channels:
                bars[i].set_color('red')
            elif channel in central_channels:
                bars[i].set_color('green')
            elif channel in parietal_channels:
                bars[i].set_color('blue')
            elif channel in frontocentral_channels:
                bars[i].set_color('brown')
        
        # Add labels and title
        ax.set_xlabel('Channels')
        ax.set_ylabel('Median Difference')
        ax.set_title(f'Median Difference between Original and Perturbed Predictions\n{freq_band.capitalize()} Band, Amplification Factor: {amp_factor}')
        
        # Rotate x-axis labels for better readability
        plt.xticks(rotation=90)
        
        # Add grid for better readability
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        
    else:
        # Create multiple subplots for all amplification factors
        amp_factors = list(median_diff_per_channel[freq_band].keys())
        fig, axs = plt.subplots(len(amp_factors), 1, figsize=(14, 4*len(amp_factors)), sharex=True)
        
        for i, factor in enumerate(sorted(amp_factors)):
            channel_diffs = median_diff_per_channel[freq_band][factor]
            
            # Filter and sort channels
            sorted_channels = [ch for ch in important_channels if ch in channel_diffs]
            sorted_values = [channel_diffs[ch] for ch in sorted_channels]
            
            # Create the bar plot
            bars = axs[i].bar(sorted_channels, sorted_values)
            
            # Color bars based on channel region
            for j, channel in enumerate(sorted_channels):
                if channel in frontal_channels:
                    bars[j].set_color('red')
                elif channel in central_channels:
                    bars[j].set_color('green')
                elif channel in parietal_channels:
                    bars[j].set_color('blue')
                elif channel in frontocentral_channels:
                    bars[j].set_color('brown')
            
            axs[i].set_ylabel('Median Difference')
            axs[i].set_title(f'Amplification Factor: {factor}')
            axs[i].grid(axis='y', linestyle='--', alpha=0.7)
            
        # Add common x-label
        fig.text(0.5, 0.04, 'Channels', ha='center', fontsize=12)
        fig.suptitle(f'Median Difference between Original and Perturbed Predictions\n{freq_band.capitalize()} Band', fontsize=16)
        plt.xticks(rotation=90)
    
    # Add a legend for the regions
    legend_elements = [
        Patch(facecolor='red', label='Frontal'),
        Patch(facecolor='green', label='Central'),
        Patch(facecolor='blue', label='Parietal'),
        Patch(facecolor='brown', label='Fronto-central')
    ]
    
    if amp_factor is not None:
        ax.legend(handles=legend_elements, loc='best')
    else:
        fig.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    
    # Save the figure if a path is provided
    if save_path:
        plt.savefig(save_path)
        
    return fig, axs if amp_factor is None else (fig, ax)


In [ ]:
plot_channel_differences(median_diff_per_channel, "alpha", amp_factor=None, save_path=None)

this looks promising, as effect of perturbation seem to be similar for neighboring channels, sugessting that potentially the model does learn spatial correlations between channels without prior information (as same perturbation in neighboring channels seem to lead to a similar effect)

make separate topoplots for each frequency band, only use one amplification factor as once again the magnitude of the factor does matter for the direction or relative strength of the perturbation

# topoplots

In [ ]:
import mne
from mne import create_info
from mne.viz import plot_topomap
import numpy as np

import matplotlib.pyplot as plt

def plot_topomap_for_freqbands(median_diff_per_channel, info=None, amp_factor=2.0, save_path=None, show_titles=False):
    """
    Create topoplots for each frequency band showing the median differences from channel perturbations.
    
    Parameters:
    -----------
    median_diff_per_channel : dict
        Dictionary containing median differences for each frequency band, amplification factor, and channel
    info : mne.Info
        MNE info object containing channel positions
    amp_factor : float
        The amplification factor to plot
    save_path : str, optional
        Path to save the figure. If None, the figure is not saved.
    show_titles : bool
        Whether to show subplot titles
    """
    # Get frequency bands
    freq_bands_list = list(median_diff_per_channel.keys())
    
    # Create a figure with subplots for each frequency band with reduced spacing
    fig, axes = plt.subplots(1, len(freq_bands_list), figsize=(4*len(freq_bands_list), 4))
    plt.subplots_adjust(wspace=-0.6)  # Reduce horizontal space between subplots
    
    for i, band in enumerate(freq_bands_list):
        # Get data for this frequency band and amplification factor
        channel_diffs = median_diff_per_channel[band][amp_factor]
        
        # Create data array in the correct order for MNE
        data = np.zeros(len(ch_names))
        for j, ch in enumerate(ch_names):
            if ch in channel_diffs:
                data[j] = channel_diffs[ch]
        
        # Plot topomap
        im, _ = plot_topomap(data, info, axes=axes[i], show=False, ch_type='eeg', 
                           contours=4, outlines='head', image_interp='nearest')
                          
        if show_titles:
            axes[i].set_title(f"{band.capitalize()}", fontsize=16)
        
        # Add colorbar with consistent size and position
        cb = plt.colorbar(im, ax=axes[i], shrink=0.5, location='bottom', pad=0.05)
        cb.set_label('Median Difference', fontsize=12)
        cb.ax.tick_params(labelsize=10)
    
    plt.tight_layout()
    if show_titles:
        plt.suptitle(f'Effect of Channel Perturbation on Model Predictions', 
                y=1.05, fontsize=18)
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
    
    return fig, axes

# Plot for amplification factor 2.0
plot_topomap_for_freqbands(median_diff_per_channel, amp_factor=1.5, info=info,
                        save_path="topo_diffs_factor_1.5.png", show_titles=True)

In [ ]:
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    pred_label_original,_,_,ch_names, info = load_predicted_amplitude_for_subject(subject_index)
    distances = load_distances(subject_index, amplification_factors
                               , rep=1)
    mean_diff_per_channel, median_diff_per_channel = calculate_diff_per_channel_dw(pred_label_original, freq_bands, amplification_factors, ch_names, distances, subject_index=subject_index)
    #plot_channel_differences(median_diff_per_channel, 'theta', 2.0, 
    #                     save_path=f"median_diff_alpha_factor_2_channels.png")
    plot_topomap_for_freqbands(median_diff_per_channel, amp_factor=1.5, info=info,
                        save_path=f"topo_diffs_factor_2.0_subject_{subject_index}.png")

In [ ]:
import mne
from mne import create_info
from mne.viz import plot_topomap
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

def plot_topomap_for_freqbands_norm(median_diff_per_channel, info=None, amp_factor=2.0, save_path=None, show_titles=False):
    """
    Create topoplots for each frequency band showing the median differences from channel perturbations.
    Data is normalized within each frequency band to better visualize patterns.
    
    Parameters:
    -----------
    median_diff_per_channel : dict
        Dictionary containing median differences for each frequency band, amplification factor, and channel
    info : mne.Info
        MNE info object containing channel positions
    amp_factor : float
        The amplification factor to plot
    save_path : str, optional
        Path to save the figure. If None, the figure is not saved.
    show_titles : bool
        Whether to show subplot titles
    """
    # Get frequency bands
    freq_bands_list = list(median_diff_per_channel.keys())
    
    # Create a figure with subplots for each frequency band with reduced spacing
    fig, axes = plt.subplots(1, len(freq_bands_list), figsize=(4*len(freq_bands_list), 4))
    plt.subplots_adjust(wspace=-0.6)  # Reduce horizontal space between subplots
    
    # For storing global min/max for consistent color scale
    all_normalized_values = []
    
    # First pass to collect data for normalization
    band_data = {}
    for band in freq_bands_list:
        # Get data for this frequency band and amplification factor
        channel_diffs = median_diff_per_channel[band][amp_factor]
        
        # Create data array in the correct order for MNE
        data = np.zeros(len(ch_names))
        for j, ch in enumerate(ch_names):
            if ch in channel_diffs:
                data[j] = channel_diffs[ch]
        
        band_data[band] = data
        
    # Normalize each band's data using Min-Max scaling within the band
    for i, band in enumerate(freq_bands_list):
        data = band_data[band]
        
        # Apply normalization - robust scaling to handle outliers
        # clip outliers to better show effect of less important channels
        q_low = np.percentile(data, 2)
        q_high = np.percentile(data, 98)
        filtered_data = np.clip(data, q_low, q_high)
        
        # Min-max scale to [0, 1]
        scaler = MinMaxScaler()
        # Reshape for the scaler
        reshaped_data = filtered_data.reshape(-1, 1)
        normalized = scaler.fit_transform(reshaped_data).flatten()
        
        # Center around zero for better visualization
        normalized = normalized * 2 - 1
        
        # Plot topomap
        im, _ = plot_topomap(normalized, info, axes=axes[i], show=False, ch_type='eeg', 
                           contours=4, outlines='head', image_interp='nearest')
        
        if show_titles:
            axes[i].set_title(f"{band.capitalize()}", fontsize=16)
        
        # Add colorbar with consistent size and position
        cb = plt.colorbar(im, ax=axes[i], shrink=0.5, location='bottom', pad=0.05)
        cb.set_label('Normalized Effect', fontsize=12)
        cb.ax.tick_params(labelsize=10)
    
    plt.tight_layout()
    if show_titles:
        plt.suptitle(f'Normalized Effect of Channel Perturbation on Model Predictions', 
                y=1.05, fontsize=18)
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
    
    return fig, axes

# Plot for amplification factor 2.0
plot_topomap_for_freqbands_norm(median_diff_per_channel, amp_factor=2.0, info=info,
                        save_path="topo_diffs_factor_2.0_normalized.png", show_titles=True)

In [ ]:
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    pred_label_original,_,_,ch_names, info = load_predicted_amplitude_for_subject(subject_index)
    mean_diff_per_channel, median_diff_per_channel = calculate_diff_per_channel(pred_label_original, freq_bands, amplification_factors, ch_names, subject_index=subject_index)
    #plot_channel_differences(median_diff_per_channel, 'theta', 2.0, 
    #                     save_path=f"median_diff_alpha_factor_2_channels.png")
    plot_topomap_for_freqbands_norm(median_diff_per_channel, amp_factor=2.0, info=info,
                        save_path=f"topo_diffs_factor_2.0_normalized_subject_{subject_index}.png")

beta and gamma band very often consider the same channels to be important BUT: The direction of importanceis always reversed

aggregation over subjects does not seem reasonable as the channels that are important differ too much.


increasing beta power in a channel often leads to a decrease in predicted amplitude, increasing gamma power often leads to an increase in predicted amplitude.

# correlations

## beta-gamma

In [ ]:
from scipy.stats import pearsonr, spearmanr
import seaborn as sns
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

cfg = load_config()

In [ ]:
def analyze_band_correlations(subjects_list, amplification_factors,freq_bands_tested=['beta', 'gamma'], amp_factor=1.5, take_abs=True):
    """
    Analyzes the correlation between beta and gamma bands across all subjects.
    
    Parameters:
    -----------
    subjects_list : list
        List of subject indices
    freq_bands_tested : list
        List of frequency bands to compare (default: beta and gamma)
    amp_factor : float
        Amplification factor to analyze
        
    Returns:
    --------
    DataFrame with correlation results for all subjects
    """
    # Initialize results storage
    results_df = pd.DataFrame(columns=[
        'Subject', 'Pearson_r', 'Pearson_p', 'Spearman_rho', 'Spearman_p', 
        'Num_Channels', 'Correlation'])
    
    # Store all subject data for scatter plot
    all_subjects_data = []
    
    # Process each subject
    for i, subject_idx in enumerate(subjects_list):
        # Load data for this subject
        distances = load_distances(subject_idx, amplification_factors, rep=1)
        pred_label_original, _, _, ch_names, info = load_predicted_amplitude_for_subject(subject_idx)
        mean_diff, median_diff = calculate_diff_per_channel_dw(
            pred_label_original, freq_bands, amplification_factors, ch_names, distances, subject_index=subject_idx,
        take_abs=take_abs)
        
        # Get values for both bands
        band1_values = median_diff[freq_bands_tested[0]][amp_factor]
        band2_values = median_diff[freq_bands_tested[1]][amp_factor]
        
        # Get common channels
        common_channels = list(set(band1_values.keys()).intersection(set(band2_values.keys())))
        
        # Extract values for comparison
        x_values = [band1_values[ch] for ch in common_channels]
        y_values = [band2_values[ch] for ch in common_channels]
        
        # Compute correlations
        pearson_r, pearson_p = pearsonr(x_values, y_values)
        spearman_rho, spearman_p = spearmanr(x_values, y_values)
        
        # Determine correlation type
        if spearman_p < 0.05:
            if spearman_rho < 0:
                corr_type = "Significant Negative"
            else:
                corr_type = "Significant Positive"
        else:
            corr_type = "Not Significant"
        
        # Store results
        results_df = results_df._append({
            'Subject': subject_idx,
            'Pearson_r': pearson_r,
            'Pearson_p': pearson_p,
            'Spearman_rho': spearman_rho,
            'Spearman_p': spearman_p,
            'Num_Channels': len(common_channels),
            'Correlation': corr_type
        }, ignore_index=True)
        
        # Store data for scatter plot
        for ch_idx, ch in enumerate(common_channels):
            all_subjects_data.append({
                'Subject': subject_idx,
                f'{freq_bands_tested[0]}': x_values[ch_idx],
                f'{freq_bands_tested[1]}': y_values[ch_idx],
                'Channel': ch,
                'Correlation': corr_type
            })
    
    # Create comprehensive dataframe for visualization
    all_data_df = pd.DataFrame(all_subjects_data)
    
    return results_df, all_data_df

def visualize_band_correlations(results_df, all_data_df, freq_bands_tested=['beta', 'gamma']):
    """
    Creates visualizations of correlation results between frequency bands.
    
    Parameters:
    -----------
    results_df : DataFrame
        Results from analyze_band_correlations function
    all_data_df : DataFrame
        All data points from all subjects
    freq_bands_tested : list
        The two frequency bands being compared
    """
    # Summary of correlations
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Distribution of correlation coefficients
    sns.histplot(results_df['Spearman_rho'], kde=True, ax=axes[0])
    axes[0].axvline(0, color='red', linestyle='--')
    axes[0].set_title(f'Distribution of Spearman Correlation between {freq_bands_tested[0].capitalize()} and {freq_bands_tested[1].capitalize()}')
    axes[0].set_xlabel('Spearman Correlation Coefficient')
    
    # Count of correlation types
    corr_counts = results_df['Correlation'].value_counts().reset_index()
    corr_counts.columns = ['Correlation Type', 'Count']
    palette = {'Significant Negative': 'red', 'Significant Positive': 'green', 'Not Significant': 'gray'}
    sns.barplot(x='Correlation Type', y='Count', data=corr_counts, palette=palette, ax=axes[1])
    axes[1].set_title('Types of Correlations Across Subjects')
    
    plt.tight_layout()
    plt.savefig(f'correlation_summary_{freq_bands_tested[0]}_{freq_bands_tested[1]}.png', dpi=300, bbox_inches='tight')
    
    # Scatter plots for all subjects with significant correlations
    sig_subjects = results_df[results_df['Correlation'] != 'Not Significant']['Subject'].values
    
    if len(sig_subjects) > 0:
        fig_scatter = plt.figure(figsize=(15, 10))
        for i, subj_idx in enumerate(sig_subjects[:min(9, len(sig_subjects))]):
            ax = fig_scatter.add_subplot(3, 3, i+1)
            subj_data = all_data_df[all_data_df['Subject'] == subj_idx]
            
            corr_type = results_df[results_df['Subject'] == subj_idx]['Correlation'].values[0]
            color = 'red' if corr_type == 'Significant Negative' else 'green'
            
            scatter = sns.scatterplot(
                x=freq_bands_tested[0], y=freq_bands_tested[1], 
                data=subj_data, 
                ax=ax, color=color, alpha=0.6
            )
            
            # Add regression line
            sns.regplot(
                x=freq_bands_tested[0], y=freq_bands_tested[1], 
                data=subj_data, 
                scatter=False, ax=ax, color=color
            )
            
            # Add correlation info
            rho = results_df[results_df['Subject'] == subj_idx]['Spearman_rho'].values[0]
            p = results_df[results_df['Subject'] == subj_idx]['Spearman_p'].values[0]
            ax.set_title(f'Subject {subj_idx} (rho={rho:.3f}, p={p:.3f})')
            
            # Add channel labels for key points
            if i == 0:  # Only for the first plot to avoid clutter
                for _, row in subj_data.sort_values(by=[freq_bands_tested[0]], ascending=False).head(3).iterrows():
                    ax.annotate(row['Channel'], (row[freq_bands_tested[0]], row[freq_bands_tested[1]]))
                for _, row in subj_data.sort_values(by=[freq_bands_tested[1]], ascending=False).head(3).iterrows():
                    ax.annotate(row['Channel'], (row[freq_bands_tested[0]], row[freq_bands_tested[1]]))
        
        plt.suptitle(f'Relationship between {freq_bands_tested[0].capitalize()} and {freq_bands_tested[1].capitalize()} Band Effects', fontsize=16)
        plt.tight_layout(rect=[0, 0, 1, 0.97])
        plt.savefig(f'scatter_plots_{freq_bands_tested[0]}_{freq_bands_tested[1]}.png', dpi=300, bbox_inches='tight')
    
    return fig

 #Run the analysis
#results_df, all_data_df = analyze_band_correlations(cfg.dataset.test_subject_indices)
#fig = visualize_band_correlations(results_df, all_data_df)

 #Print summary statistics
#print(f"Total subjects analyzed: {len(results_df)}")
#print(f"Subjects with significant negative correlation: {len(results_df[results_df['Correlation'] == 'Significant Negative'])}")
#print(f"Subjects with significant positive correlation: {len(results_df[results_df['Correlation'] == 'Significant Positive'])}")
#print(f"Subjects with no significant correlation: {len(results_df[results_df['Correlation'] == 'Not Significant'])}")
#print(f"\nAverage correlation across all subjects: {results_df['Spearman_rho'].mean():.4f}")


In [ ]:
def visualize_band_correlations(results_df, all_data_df, freq_bands_tested=['beta', 'gamma']):
    """
    Creates visualizations of correlation results between frequency bands.
    
    Parameters:
    -----------
    results_df : DataFrame
        Results from analyze_band_correlations function
    all_data_df : DataFrame
        All data points from all subjects
    freq_bands_tested : list
        The two frequency bands being compared
    """
    # Summary of correlations
    fig, axes = plt.subplots(1, 1, figsize=(8, 8))
    
    # Distribution of correlation coefficients
    sns.histplot(results_df['Pearson_r'], kde=True, ax=axes)
    #axes[0].axvline(0, color='red', linestyle='--')
    axes.set_title(f'Distribution of Pearson Correlation between {freq_bands_tested[0].capitalize()} and {freq_bands_tested[1].capitalize()}', fontsize=18)
    axes.set_xlabel('Pearson Correlation Coefficient', fontsize=18)
    axes.set_ylabel('Count', fontsize=18)
    axes.tick_params(axis='both', labelsize=16)

    

    
    return fig

#Run the analysis
#cfg = load_config()
#results_df, all_data_df = analyze_band_correlations(cfg.dataset.test_subject_indices)
#fig = visualize_band_correlations(results_df, all_data_df)



In [ ]:
#fig.savefig("correlation_summary_pearson_beta_gamma.png", dpi=300, bbox_inches='tight')

## all correlations

In [ ]:
from scipy.stats import pearsonr, spearmanr
import seaborn as sns
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

def analyze_all_band_correlations(subjects_list, amp_factor=180):
    """
    Analyzes correlations between all possible pairs of frequency bands across subjects.
    
    Parameters:
    -----------
    subjects_list : list
        List of subject indices
    amp_factor : float
        Amplification factor to analyze
        
    Returns:
    --------
    Dictionary with results for each pair of frequency bands
    """
    # Get all frequency band pairs
    freq_band_list = list(freq_bands.keys())
    band_pairs = []
    for i in range(len(freq_band_list)):
        for j in range(i+1, len(freq_band_list)):
            band_pairs.append((freq_band_list[i], freq_band_list[j]))
    
    # Initialize results storage
    all_results = {}
    
    # Process each band pair
    for band_pair in band_pairs:
        band1, band2 = band_pair
        print(f"Analyzing {band1} vs {band2}...")
        
        # Get results for this pair
        results_df, all_data_df = analyze_band_correlations(
            subjects_list, 
            freq_bands_tested=[band1, band2], 
            amp_factor=amp_factor
        )
        
        # Store results
        all_results[f"{band1}_{band2}"] = {
            'results_df': results_df, 
            'all_data_df': all_data_df
        }
        
        # Visualize this pair
        fig = visualize_band_correlations(results_df, all_data_df, freq_bands_tested=[band1, band2])
        plt.close()
    
    # Create summary of all pairs
    summary_data = []
    for pair_name, pair_results in all_results.items():
        band1, band2 = pair_name.split('_')
        results = pair_results['results_df']
        
        # Calculate summary statistics
        sig_neg = len(results[results['Correlation'] == 'Significant Negative'])
        sig_pos = len(results[results['Correlation'] == 'Significant Positive'])
        not_sig = len(results[results['Correlation'] == 'Not Significant'])
        avg_corr = results['Spearman_rho'].mean()
        
        summary_data.append({
            'Band Pair': f"{band1}-{band2}",
            'Significant Negative': sig_neg,
            'Significant Positive': sig_pos,
            'Not Significant': not_sig,
            'Average Correlation': avg_corr,
            'Total Subjects': len(results)
        })
    
    # Create summary dataframe
    summary_df = pd.DataFrame(summary_data)
    
    return all_results, summary_df

# Run the analysis for all pairs


# Create bar chart showing number of significant correlations
#summary_for_plot = summary_df.copy()
#summary_for_plot = pd.melt(
#    summary_for_plot, 
#    id_vars=['Band Pair'], 
#    value_vars=['Significant Negative', 'Significant Positive', 'Not Significant'],
#    var_name='Correlation Type', 
#    value_name='Count'
#)

#plt.figure(figsize=(16, 8))
#ax = sns.barplot(
#    x='Band Pair', 
#    y='Count', 
#    hue='Correlation Type', 
#    data=summary_for_plot,
#    palette={'Significant Negative': 'red', 'Significant Positive': 'green', 'Not Significant': 'gray'}
#)
#plt.xticks(rotation=45)
#plt.title('Distribution of Correlation Types Between Frequency Bands')
#plt.tight_layout()
#plt.savefig('correlation_types_distribution.png', dpi=300, bbox_inches='tight')
#plt.show()

In [ ]:
all_results, summary_df = analyze_all_band_correlations(cfg.dataset.test_subject_indices)

# Display summary results
print("\n==== SUMMARY OF BAND CORRELATIONS ====\n")
print(summary_df.sort_values(by='Average Correlation'))

# Create summary visualization
plt.figure(figsize=(8, 8))

# Create heatmap of average correlations
corr_matrix = np.zeros((len(freq_bands), len(freq_bands)))
labels = list(freq_bands.keys())

# Fill correlation matrix
for i, band1 in enumerate(labels):
    for j, band2 in enumerate(labels):
        if i == j:
            corr_matrix[i, j] = 1.0  # Diagonal is 1.0
        else:
            pair_name = f"{band1}_{band2}" if i < j else f"{band2}_{band1}"
            if pair_name in all_results:
                corr_matrix[i, j] = all_results[pair_name]['results_df']['Spearman_rho'].mean()

# Create heatmap
ax = sns.heatmap(corr_matrix, 
                 annot=True, 
                 fmt=".2f",
                 cmap='coolwarm', 
                 vmin=-1, vmax=1,
                 xticklabels=[band.capitalize() for band in labels],
                 yticklabels=[band.capitalize() for band in labels], cbar=False, annot_kws={'size': 15})

plt.title('Average Spearman Correlation Between Frequency Bands')
plt.tight_layout()
plt.savefig('all_frequency_band_correlations_abs.png', dpi=300, bbox_inches='tight')
plt.show()

from scipy.stats import pearsonr, spearmanr
import seaborn as sns
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

def analyze_all_band_correlations(subjects_list, amp_factor=1.5):

  
    freq_band_list = list(freq_bands.keys())
    band_pairs = []
    for i in range(len(freq_band_list)):
        for j in range(i+1, len(freq_band_list)):
            band_pairs.append((freq_band_list[i], freq_band_list[j]))
    
   
    all_results = {}
    

    for band_pair in band_pairs:
        band1, band2 = band_pair
      
        
        
        results_df, all_data_df = analyze_band_correlations(
            subjects_list, 
            phase_peturbations,
            freq_bands_tested=[band1, band2], 
            amp_factor=amp_factor
        )
  
        all_results[f"{band1}_{band2}"] = {
            'results_df': results_df, 
            'all_data_df': all_data_df
        }

        #fig = visualize_band_correlations(results_df, all_data_df, freq_bands_tested=[band1, band2])
        #plt.close()
  
    summary_data = []
    for pair_name, pair_results in all_results.items():
        band1, band2 = pair_name.split('_')
        results = pair_results['results_df']
    
        sig_neg = len(results[results['Correlation'] == 'Significant Negative'])
        sig_pos = len(results[results['Correlation'] == 'Significant Positive'])
        not_sig = len(results[results['Correlation'] == 'Not Significant'])
        avg_corr = results['Pearson_r'].mean() 
        
        summary_data.append({
            'Band Pair': f"{band1}-{band2}",
            'Significant Negative': sig_neg,
            'Significant Positive': sig_pos,
            'Not Significant': not_sig,
            'Average Correlation': avg_corr,
            'Total Subjects': len(results)
        })
    
    
    summary_df = pd.DataFrame(summary_data)
    
    return all_results, summary_df

all_results, summary_df = analyze_all_band_correlations(cfg.dataset.test_subject_indices, amp_factor=180)


print("\n==== SUMMARY OF BAND CORRELATIONS ====\n")
print(summary_df.sort_values(by='Average Correlation'))



corr_matrix = np.zeros((len(freq_bands), len(freq_bands)))
labels = list(freq_bands.keys())

for i, band1 in enumerate(labels):
    for j, band2 in enumerate(labels):
        if i == j:
            corr_matrix[i, j] = 1.0  
        else:
            pair_name = f"{band1}_{band2}" if i < j else f"{band2}_{band1}"
            if pair_name in all_results:
                corr_matrix[i, j] = all_results[pair_name]['results_df']['Pearson_r'].mean() 
fig, axs = plt.subplots(figsize=(8,8))

ax = sns.heatmap(corr_matrix, 
                 annot=True, 
                 fmt=".2f",
                 cmap='coolwarm', 
                 vmin=-1, vmax=1,
                 xticklabels=[band.capitalize() for band in labels],
                 yticklabels=[band.capitalize() for band in labels], 
                 cbar=False, 
                 annot_kws={'size': 17}, ax=axs),



axs.tick_params(axis='both', which='major', labelsize=16)


fig.suptitle('Average Correlation Between Frequency Bands', fontsize=20)
fig.savefig('phase_all_frequency_band_correlations_pearson_abs.png', dpi=300, bbox_inches='tight') 
plt.tight_layout()
plt.savefig('all_frequency_band_correlations_pearson.png', dpi=300, bbox_inches='tight') 
plt.show()

